<center><ins><h1>Next-Generation Sequencing</h1></ins></center>

<h5>Definition:</h5>
<ul>
    <li>Next-Generation Sequencing or whole genome sequencing (WGS), also known as full genome sequencing, complete genome sequencing, or entire genome sequencing, is the process of determining the entirety, or nearly the entirety, of the DNA sequence of an organism's genome at a single time.[2] This entails sequencing all of an organism's chromosomal DNA as well as DNA contained in the mitochondria and, for plants, in the chloroplast.</li>
    <li>No parameter emphasized yet</li>
    <li>Files are imported over .tsv as abundance tables for each sample (pooled from three individual experiments).</li>
</ul>

<center>
<img src="../images/ngs_device.png" alt="Device" width="400" height="300">
<img src="../images/ngs_diagram.png" alt="Diagram" width="489" height="300">
<img src="../images/ngs_example-graph.png" alt="Example-Graph" width="327" height="300">
</center>

###### **Author:** Cedric Hering-Peter
###### **Address:** AG Schulz / Botantical Institut / Am Botanischen Garten 5 / 24118 Kiel

# Packages

In [1]:
# Imports
import os, sys, warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from scripts import plot_helper
import pandas as pd

# Variables

In [2]:
if not data_helper.should_run_ngs(CONFIG):
    raise SystemExit(
        "NGS analysis is disabled in example mode. "
        "Set data_mode: full in analysis_config.yaml to run this notebook."
    )
if not data_helper.should_run_ngs(CONFIG):
    raise SystemExit(
        "NGS analysis is disabled in example mode. "
        "Set data_mode: full in analysis_config.yaml to run this notebook."
    )
RECAL_SEQS = False # Toggle for recalculation of sequencing files
TOP_AMOUNT = 10 # Amount of shown taxa in plot
IGNORE_EPS = True # Prevent +EPS data to be shown

# Custom filter for taxa
TAX_FIL = "g__"
MIC_FIL = ["Rhodophyta", "Chlorophyta", "Charophyceae", "Coleochaetophyceae", "Zygnemophyceae", 
           "Glaucophyta", "Stramenopiles", "Heterokontophyta", "Bacillariophyta", "Bolidophyceae", 
           "Dictyochophyceae", "Pelagophyceae", "Chrysophyceae", "Synurophyceae", "Eustigmatophyceae", 
           "Synchromophyceae", "Pinguiophyceae", "Raphidophyceae", "Xanthophyceae", "Phaeophyceae", 
           "Dinoflagellata", "Colpodellida", "Rhizaria", "Cercozoa", "Chlorarachniophyta", 
           "Paulinellidae", "Cryptista", "Haptista", "Euglenozoa", "Euglenida"]
CYA_FIL = ["Cyanobacteria"]
BAC_FIL = ["Bacteria"]

# Custom data ordering and naming - CAREFUL To CHANGE - Case sensitive!
SAMPLES_ORDER = {"CH230921": ["CC FEMAK - 1", "CC FEMAK - 2", "CC FEMAK - 3", "CC OECD - 1", "CC OECD - 2", "CC OECD - 3", "CC RueBio - 1", "CC RueBio - 2", "CC RueBio - 3"],
                 "CH-Physiology": ["AT Control 0-h 1", "AT Control 48-h 1", "AT 0.1 48-h 1", "AT 1 48-h 1", "AT 10 48-h 1", "AT 100 48-h 1", 
                 "LT Control 0-h 1", "LT Control 48-h 1", "LT 25 48-h 1", "LT 50 48-h 1", "LT 70 48-h 1", "LT 100 48-h 1",
                 "PT Control 0-h 1", "PT Control 48-h 1", "PT 3 48-h 1", "PT 6 48-h 1", "PT 9 48-h 1", "PT 12 48-h 1",
                 "ST Control 0-h 1", "ST Control 48-h 1", "ST 0.75 48-h 1", "ST 1.5 48-h 1", "ST 15 48-h 1", "ST 35 48-h 1",
                 "TT Control 0-h 1", "TT Control 48-h 1", "TT 5 48-h 1", "TT 15 48-h 1", "TT 35 48-h 1", "TT 45 48-h 1"],
                 "CH-Flocculation": ["RM Control 0-h 1", "RM Control 48-h 1", "RM -EPS 0-h 1", "RM -EPS 48-h 1", "RM +EPS 48-h 1", "RM AT 48-h 1", "RM ET 48-h 1", 
                                     "BM CoA 0-h 1", "BM CoA 48-h 1", "BM AMC 48-h 1"],
                 "CH230117": ["SC Control - 1"]}   
SAMPLES_FULLNAMES = {"CH230921": ["CC Sewage  1", "CC Sewage  2", "CC Sewage  3", "CC Artificial Medium  1", "CC Artificial Medium  2", "CC Artificial Medium  3", "CC Fish Sewage  1", "CC Fish Sewage  2", "CC Fish Sewage  3"],
                 "CH-Physiology": ["AT $\\bf{0}$ $\\bf{mL}$ $\\bf{(0 h)}$ 1", "AT $\\bf{0}$ $\\bf{mL}$ $\\bf{(48 h)}$ 1", "AT 0.1 mL (48 h) 1", "AT 1 mL (48 h) 1", "AT 10 mL (48 h) 1", "AT 100 mL (48 h) 1", 
                 "LT $\\bf{100}$ $\\bf{µmol}$ $\\bf{m^{\u207B2}}$ $\\bf{s^{\u207B1}}$ $\\bf{(0 h)}$ 1", "LT $\\bf{100}$ $\\bf{µmol}$ $\\bf{m^{\u207B2}}$ $\\bf{s^{\u207B1}}$ $\\bf{(48 h)}$ 1", "LT 375 µmol $m^{\u207B2}$ $s^{\u207B1}$ (48 h) 1", "LT 750 µmol $m^{\u207B2}$ $s^{\u207B1}$ (48 h) 1", "LT 1050 µmol $m^{\u207B2}$ $s^{\u207B1}$ (48 h) 1", "LT 1500 µmol $m^{\u207B2}$ $s^{\u207B1}$ (48 h) 1",
                 "PT $\\bf{pH}$ $\\bf{7}$ $\\bf{(0 h)}$ 1", "PT $\\bf{pH}$ $\\bf{7}$ $\\bf{(48 h)}$ 1", "PT pH 3 (48 h) 1", "PT pH 6 (48 h) 1", "PT pH 9 (48 h) 1", "PT pH 12 (48 h) 1",
                 "ST $\\bf{0}$ $\\bf{PSU}$ $\\bf{(0 h)}$ 1", "ST $\\bf{0}$ $\\bf{PSU}$ $\\bf{(48 h)}$ 1", "ST 0.75 PSU (48 h) 1", "ST 1.5 PSU (48 h) 1", "ST 15 PSU (48 h) 1", "ST 35 PSU (48 h) 1",
                 "TT $\\bf{25}$ $\\bf{°C}$ $\\bf{(0 h)}$ 1", "TT $\\bf{25}$ $\\bf{°C}$ $\\bf{(48 h)}$ 1", "TT 5 °C (48 h) 1", "TT 15 °C (48 h) 1", "TT 35 °C (48 h) 1", "TT 45 °C (48 h) 1"],
                 "CH-Flocculation": ["RM $\\bf{MBC}$ $\\bf{(0 h)}$ 1", "RM $\\bf{MBC}$ $\\bf{(48 h)}$ 1", "RM $\\bf{dH_2O}$ $\\bf{(0 h)}$ 1", "RM $\\bf{dH_2O}$ $\\bf{(48 h)}$ 1", "RM +EPS (48 h) 1", "RM Antibiotics (48 h) 1", "RM Enzymes (48 h) 1",
                                     "BM $\\bf{MBC}$ $\\bf{(0 h)}$ 1", "BM $\\bf{MBC}$ $\\bf{(48 h)}$ 1", "BM MBC+$\it{C. vulgaris}$ (48 h) 1"],
                 "CH230117": ["SC MBC  1"]}

# Show numeric output in decimal format e.g., 2.1514
pd.options.display.float_format = '{:,.2f}'.format

# Functions

In [3]:
def read_sample_information(file):
    # Extract sample information from file create a data frame
    sam_inf = pd.DataFrame([{"SAMPLE_INFORMATION":os.path.basename(file).split(".taxonomic_profile.tsv")[0]}])
    sam_inf[data_helper.get_sample_columns(META_DATA)] = sam_inf["SAMPLE_INFORMATION"].str.split("_", expand = True)
    sam_inf = sam_inf.drop("SAMPLE_INFORMATION", axis = 1)
    return sam_inf

def append_microalgae(sam_inf, raw_sam):
    # Filter only microalgae
    fil_sam = pd.DataFrame([(raw_sam.iloc[index].Taxonomy, raw_sam.iloc[index].Count) for index, row in raw_sam.iterrows() if any([group in row["Taxonomy"] for group in MIC_FIL])], columns = ["Taxonomy", "Count"])
    # Scan for all TAX_FIL (most commonely genuses)
    fil_sam = pd.DataFrame([(fil_sam.iloc[index].Taxonomy.split("|")[-1], fil_sam.iloc[index].Count) for index, entry in fil_sam.iterrows() if TAX_FIL in entry.Taxonomy.split("|")[-1] ], columns = ["Taxonomy", "Count"])
    # Identify and merge-sum all duplicates
    print(f"{fil_sam.Taxonomy.duplicated().values.sum()} duplicate(s) found and merged in current data frame.")
    fil_sam = fil_sam.groupby("Taxonomy").sum()
    # Merge sample information to the left of (transposed) clean taxonomy data
    fil_sam = pd.concat([sam_inf, fil_sam.T.reset_index()], axis = 1)
    fil_sam.rename(columns = {"index": "FILTER"}, inplace = True)
    fil_sam["FILTER"] = "Microalgae"
    return fil_sam
def append_cyanobacteria(sam_inf, raw_sam):
    # Filter only cyanobacteria
    fil_sam = pd.DataFrame([(raw_sam.iloc[index].Taxonomy, raw_sam.iloc[index].Count) for index, row in raw_sam.iterrows() if any([group in row["Taxonomy"] for group in CYA_FIL])], columns = ["Taxonomy", "Count"])
    # Scan for all TAX_FIL (most commonely genuses)
    fil_sam = pd.DataFrame([(fil_sam.iloc[index].Taxonomy.split("|")[-1], fil_sam.iloc[index].Count) for index, entry in fil_sam.iterrows() if TAX_FIL in entry.Taxonomy.split("|")[-1] ], columns = ["Taxonomy", "Count"])
    # Identify and merge-sum all duplicates
    print(f"{fil_sam.Taxonomy.duplicated().values.sum()} duplicate(s) found and merged in current data frame.")
    fil_sam = fil_sam.groupby("Taxonomy").sum()
    # Merge sample information to the left of (transposed) clean taxonomy data
    fil_sam = pd.concat([sam_inf, fil_sam.T.reset_index()], axis = 1)
    fil_sam.rename(columns = {"index": "FILTER"}, inplace = True)
    fil_sam["FILTER"] = "Cyanobacteria"
    return fil_sam
def append_bacteria(sam_inf, raw_sam):
    # Filter only cyanobacteria
    fil_sam = pd.DataFrame([(raw_sam.iloc[index].Taxonomy, raw_sam.iloc[index].Count) for index, row in raw_sam.iterrows() if any([group in row["Taxonomy"] for group in BAC_FIL])], columns = ["Taxonomy", "Count"])
    # Scan for all TAX_FIL (most commonely genuses)
    fil_sam = pd.DataFrame([(fil_sam.iloc[index].Taxonomy.split("|")[-1], fil_sam.iloc[index].Count) for index, entry in fil_sam.iterrows() if TAX_FIL in entry.Taxonomy.split("|")[-1] ], columns = ["Taxonomy", "Count"])
    # Identify and merge-sum all duplicates
    print(f"{fil_sam.Taxonomy.duplicated().values.sum()} duplicate(s) found and merged in current data frame.")
    fil_sam = fil_sam.groupby("Taxonomy").sum()
    # Merge sample information to the left of (transposed) clean taxonomy data
    fil_sam = pd.concat([sam_inf, fil_sam.T.reset_index()], axis = 1)
    fil_sam.rename(columns = {"index": "FILTER"}, inplace = True)
    fil_sam["FILTER"] = "Bacteria"
    return fil_sam

def create_heatmap_matrix(sub_df, id):
    # Slice pure values and sort for top 10 taxa
    pure_vals = sub_df.iloc[:, 7:].dropna(axis = 1, how = "all")
    sum = pure_vals.sum()
    pure_vals = pure_vals[sum.sort_values(ascending = False).index[:TOP_AMOUNT]]

    # Join sample information and 
    #sub_df["BIO_REP"] = sub_df["BIO_REP"].astype(str)
    sub_df.loc[:, ["BIO_REP"]] = sub_df["BIO_REP"].astype(str)
    sub_inf_df = sub_df.iloc[:, 1:5].stack().groupby(level = 0).agg(" ".join) # Cutted Sample information
    pure_vals.insert(0, "SAMPLE_NAME", sub_inf_df) # Add sample information to pure values
    plot_df = pure_vals

    # Sort after custom categories SAMPLES_ORDER
    plot_df["SAMPLE_NAME"] = pd.Categorical(plot_df["SAMPLE_NAME"], categories = SAMPLES_ORDER.get(id)) 
    plot_df = plot_df.sort_values(by = "SAMPLE_NAME")

    # Insert full names with regular expressions
    plot_df["SAMPLE_NAME"] = SAMPLES_FULLNAMES.get(id)

    # Set index to sample names and replace prefixes and appendixes
    plot_df.set_index("SAMPLE_NAME", inplace = True)
    plot_df.index = plot_df.index.str.replace("-", " ").str.replace("-", "")
    plot_df.index = [sample_name[3:-2] for sample_name in plot_df.index]
    plot_df.columns = plot_df.columns.str.replace("g__", "")

    # Calculate percentage relative to total top10 microalgae in sample
    plot_df = plot_df[plot_df.columns].div(plot_df[plot_df.columns].sum(axis = 1), axis = 0).multiply(100) 
    return plot_df

# Raw Import

In [4]:
# Define data path and get all associated files
files_dir = data_helper.get_instrument_path(CONFIG, "ngs")

# Open each file and extract microalgae, cyanobacteria and bacteria taxa counts into new data frame
raw_df = pd.DataFrame()
for dir_path, dirs, files in os.walk(files_dir):
    if not RECAL_SEQS: break # Save time and import from .csv
    for file in files:
        if file.endswith("taxonomic_profile.tsv"):
            print(f"Working on: {file}")
            sam_inf = read_sample_information(file)
            # Read raw sample data
            file_path = os.path.join(dir_path, file)
            raw_sam = pd.read_csv(file_path, sep = "\t", names = ["Taxonomy", "Count"])
            # Append filtered taxa to raw data frame
            raw_df = pd.concat([raw_df, 
                                append_microalgae(sam_inf, raw_sam), 
                                append_cyanobacteria(sam_inf, raw_sam), 
                                append_bacteria(sam_inf, raw_sam)], ignore_index = True)

# Fast Raw Import from CSV

In [5]:
# Import from .csv to save time
if not RECAL_SEQS:   
    raw_df = pd.read_csv(data_helper.get_output_path(CONFIG, "Raw-Data/Taxa-Distribution/Taxa-Distribution_Raw-Data.csv")"))

# Convert NaN values to zeros and all numbers into integer 
raw_df.iloc[:, 6:] = raw_df.iloc[:, 6:].fillna(0)

if IGNORE_EPS:
    raw_df.drop(raw_df[raw_df["SAMPLE_NAME"] == "+EPS"].index, inplace = True)
    SAMPLES_ORDER["CH-Flocculation"] = [sample_index for sample_index in SAMPLES_ORDER["CH-Flocculation"] if sample_index != 4]
    SAMPLES_FULLNAMES["CH-Flocculation"] = [sample_appendix for index, sample_appendix in enumerate(SAMPLES_FULLNAMES["CH-Flocculation"]) if index != 4]

print(f"Shape: {raw_df.shape}")
raw_df.sample()

Shape: (147, 3823)


,ID,EXPERIMENT_NAME,SAMPLE_NAME,TIME,BIO_REP,TEC_REP,FILTER,g__Acanthoceras,g__Achnanthes,g__Achnanthidium,...,g__Ulvaria,g__Acetitomaculum,g__Allofustis,g__Bavariicoccus,g__Cyanodorina,g__Ereboglobus,g__Muribacter,g__Salsuginibacillus,g__Winkia,g__Wohlfahrtiimonas
45,CH-Physiology,PT,Control,0-h,1,1,Microalgae,5.00,0.00,1.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [6]:
cyanos_only = raw_df[raw_df["FILTER"] == "Cyanobacteria"]
summed_cyanos = cyanos_only.iloc[:, 7:].sum(axis = 1)
meta_cyanos = cyanos_only.iloc[:, :6]
columns = ["ID", "EXPERIMENT_NAME", "SAMPLE_NAME", "TIME", "BIO_REP", "TEC_REP", "COUNT"]
export_cyano = pd.concat([meta_cyanos, summed_cyanos], axis = 1)
export_cyano.columns = columns

# File Export

In [7]:
# Export to raw data to csv-file
export_df = pd.DataFrame(raw_df)
file_name = data_helper.get_output_path(CONFIG, "Raw-Data/Taxa-Distribution/Taxa-Distribution_Raw-Data.csv")
export_df.to_csv(file_name, encoding = "utf-8", index = False)

# Export summed cyano data
file_name = data_helper.get_output_path(CONFIG, "Raw-Data/Taxa-Distribution/Cyano-Counts_Raw-Data.csv")
export_cyano.to_csv(file_name, encoding = "utf-8", index = False)

# Plot Visualization

In [8]:
complete_mic_df = pd.DataFrame()

for id in raw_df["ID"].unique():
    # Prepare data frame for each plot
    mic_df = create_heatmap_matrix(raw_df[(raw_df["ID"] == id) & (raw_df["FILTER"] == "Microalgae")], id)
    cya_df = create_heatmap_matrix(raw_df[(raw_df["ID"] == id) & (raw_df["FILTER"] == "Cyanobacteria")], id)
    bac_df = create_heatmap_matrix(raw_df[(raw_df["ID"] == id) & (raw_df["FILTER"] == "Bacteria")], id)
    
    # Turn genuses into italic font and add appendix
    mic_df.columns = ["$\it{" + name + "}$ spp." for name in list(mic_df.columns)]
    cya_df.columns = ["$\it{" + name + "}$ spp." for name in list(cya_df.columns)]
    bac_df.columns = ["$\it{" + name + "}$ spp." for name in list(bac_df.columns)]
    
    # Export Top 10 Data
    exp_df = pd.concat([mic_df, cya_df, bac_df], axis = 1)
    file_name = data_helper.get_output_path(CONFIG, "Raw-Data/Taxa-Distribution/Taxa-Distribution_" + id + "_Top10-Data.csv")
    exp_df.to_csv(file_name, encoding = "utf-8", index = False)
    
    plot_helper.visualize_heatmaps(mic_df, 
                                   cya_df, 
                                   bac_df, 
                                   id)

Heatmap "CH-Physiology" created and saved.
Heatmap "CH230921" created and saved.
Heatmap "CH-Flocculation" created and saved.
Heatmap "CH230117" created and saved.


In [9]:
import matplotlib
print(matplotlib.matplotlib_fname())

/Users/cedric/Documents/Career/PhD-Student/Experiments/Data-Science/lib/python3.11/site-packages/matplotlib/mpl-data/matplotlibrc


# ARCHIVE